# Bitcoin Trading Performance & Market Sentiment Analysis
## Primetrade.ai Data Science Task

This notebook explores the relationship between trader performance on Hyperliquid and Bitcoin market sentiment (Fear & Greed Index).

**Analysis Goals:**
1. Understand how market sentiment affects trading outcomes
2. Identify hidden patterns in trader behavior
3. Develop insights for smarter trading strategies

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

## 1. Data Loading and Preprocessing

In [ ]:
# Load datasets
fear_greed = pd.read_csv('fear_greed_index.csv')
historical = pd.read_csv('historical_data.csv')

print(f"Fear/Greed Index: {len(fear_greed):,} records")
print(f"Historical Trading Data: {len(historical):,} records")

In [ ]:
# Preview datasets
print("Fear/Greed Index Sample:")
display(fear_greed.head())

print("\nHistorical Trading Data Sample:")
display(historical.head())

In [ ]:
# Data preprocessing
fear_greed['date'] = pd.to_datetime(fear_greed['date'])
historical['Timestamp'] = pd.to_datetime(historical['Timestamp IST'], format='%d-%m-%Y %H:%M', errors='coerce')
historical['date'] = historical['Timestamp'].dt.normalize()

# Convert numerical columns
historical['Closed PnL'] = pd.to_numeric(historical['Closed PnL'], errors='coerce')
historical['Size USD'] = pd.to_numeric(historical['Size USD'], errors='coerce')
historical['Execution Price'] = pd.to_numeric(historical['Execution Price'], errors='coerce')

print("Data types after conversion:")
print(historical[['Timestamp', 'Closed PnL', 'Size USD']].dtypes)

In [ ]:
# Merge datasets on date
merged = historical.merge(
    fear_greed[['date', 'classification', 'value']], 
    on='date', 
    how='left'
)

print(f"Merged dataset: {len(merged):,} records")
print(f"Date range: {merged['date'].min()} to {merged['date'].max()}")
print(f"\nMissing sentiment data: {merged['classification'].isna().sum():,} rows")

## 2. Exploratory Data Analysis

In [ ]:
# Basic statistics
print("Fear/Greed Classification Distribution:")
print(fear_greed['classification'].value_counts())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

fear_greed['classification'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Market Sentiment Distribution', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Sentiment')
axes[0].set_ylabel('Number of Days')
axes[0].grid(True, alpha=0.3)

historical['Side'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%')
axes[1].set_title('Buy vs Sell Distribution', fontsize=12, fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
# Top traded assets
print("Top 10 Most Traded Coins:")
top_coins = historical['Coin'].value_counts().head(10)
print(top_coins)

plt.figure(figsize=(12, 6))
top_coins.plot(kind='barh', color='purple', alpha=0.7)
plt.title('Top 10 Most Traded Coins', fontsize=14, fontweight='bold')
plt.xlabel('Number of Trades')
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## 3. Sentiment Impact Analysis

In [ ]:
# Performance metrics by sentiment
sentiment_metrics = merged.groupby('classification').agg({
    'Closed PnL': ['mean', 'median', 'sum', 'count'],
    'Size USD': ['mean', 'sum'],
    'Account': 'nunique'
}).round(2)

print("Performance Metrics by Market Sentiment:")
display(sentiment_metrics)

In [ ]:
# Win rate by sentiment
win_rate = merged.groupby('classification').apply(
    lambda x: (x['Closed PnL'] > 0).mean() * 100
).sort_values(ascending=False)

print("Win Rate by Sentiment:")
print(win_rate)

plt.figure(figsize=(10, 6))
win_rate.plot(kind='bar', color='green', alpha=0.7)
plt.title('Win Rate (%) by Market Sentiment', fontsize=14, fontweight='bold')
plt.ylabel('Win Rate (%)')
plt.xlabel('Sentiment')
plt.xticks(rotation=45, ha='right')
plt.axhline(y=50, color='red', linestyle='--', label='50% Breakeven')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Average PnL by sentiment
avg_pnl = merged.groupby('classification')['Closed PnL'].mean().sort_values()

plt.figure(figsize=(10, 6))
colors = ['red' if x < 0 else 'green' for x in avg_pnl.values]
avg_pnl.plot(kind='barh', color=colors, alpha=0.7)
plt.title('Average PnL by Market Sentiment', fontsize=14, fontweight='bold')
plt.xlabel('Average Closed PnL ($)')
plt.axvline(x=0, color='black', linestyle='-', linewidth=1)
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## 4. Temporal Pattern Analysis

In [ ]:
# Extract time features
merged['hour'] = merged['Timestamp'].dt.hour
merged['day_of_week'] = merged['Timestamp'].dt.day_name()

# Hourly performance
hourly_pnl = merged.groupby('hour')['Closed PnL'].mean()

plt.figure(figsize=(14, 6))
plt.plot(hourly_pnl.index, hourly_pnl.values, marker='o', linewidth=2, markersize=8, color='darkgreen')
plt.fill_between(hourly_pnl.index, 0, hourly_pnl.values, alpha=0.3, color='green')
plt.title('Average PnL by Hour of Day (UTC)', fontsize=14, fontweight='bold')
plt.xlabel('Hour')
plt.ylabel('Average Closed PnL ($)')
plt.axhline(y=0, color='red', linestyle='--', linewidth=1)
plt.grid(True, alpha=0.3)
plt.xticks(range(24))
plt.tight_layout()
plt.show()

print("\nBest performing hours:")
print(hourly_pnl.nlargest(5))

## 5. Trader Performance Analysis

In [ ]:
# Aggregate trader performance
trader_performance = merged.groupby('Account').agg({
    'Closed PnL': 'sum',
    'Size USD': 'sum',
    'Account': 'count'
}).rename(columns={'Account': 'trade_count'})

trader_performance['avg_pnl_per_trade'] = (
    trader_performance['Closed PnL'] / trader_performance['trade_count']
)

print(f"Total Unique Traders: {len(trader_performance)}")
print(f"Profitable Traders: {(trader_performance['Closed PnL'] > 0).sum()}")
print(f"Loss-making Traders: {(trader_performance['Closed PnL'] < 0).sum()}")

print("\nTop 10 Traders by Total PnL:")
display(trader_performance.nlargest(10, 'Closed PnL'))

In [ ]:
# Visualize trader PnL distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top traders
top_15 = trader_performance.nlargest(15, 'Closed PnL')['Closed PnL'] / 1000
top_15.plot(kind='barh', ax=axes[0], color='darkgreen')
axes[0].set_title('Top 15 Traders by Total PnL', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Total PnL (Thousands $)')
axes[0].grid(True, alpha=0.3, axis='x')

# PnL distribution
axes[1].hist(merged['Closed PnL'].dropna(), bins=100, color='steelblue', alpha=0.7, edgecolor='black')
axes[1].set_title('Distribution of Individual Trade PnL', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Closed PnL ($)')
axes[1].set_ylabel('Frequency')
axes[1].axvline(x=0, color='red', linestyle='--', linewidth=2, label='Breakeven')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Advanced Pattern Detection

In [ ]:
# Buy vs Sell performance by sentiment
side_sentiment = merged.groupby(['classification', 'Side'])['Closed PnL'].mean().unstack()

print("Average PnL: Buy vs Sell by Sentiment")
display(side_sentiment)

side_sentiment.plot(kind='bar', figsize=(12, 6), width=0.8)
plt.title('Average PnL: Buy vs Sell by Sentiment', fontsize=14, fontweight='bold')
plt.ylabel('Average Closed PnL ($)')
plt.xlabel('Sentiment')
plt.xticks(rotation=45, ha='right')
plt.axhline(y=0, color='red', linestyle='--', linewidth=1)
plt.legend(title='Side')
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# Trade size quartile analysis
merged['size_quartile'] = pd.qcut(
    merged['Size USD'].dropna(), 
    q=4, 
    labels=['Q1-Small', 'Q2-Medium', 'Q3-Large', 'Q4-XLarge']
)

size_performance = merged.groupby('size_quartile').agg({
    'Closed PnL': ['mean', lambda x: (x > 0).mean() * 100]
}).round(2)

print("Performance by Trade Size Quartile:")
display(size_performance)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

size_performance[('Closed PnL', 'mean')].plot(kind='bar', ax=axes[0], color='teal')
axes[0].set_title('Average PnL by Trade Size', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Average PnL ($)')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')
axes[0].grid(True, alpha=0.3, axis='y')

size_performance[('Closed PnL', '<lambda_0>')].plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Win Rate by Trade Size', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Win Rate (%)')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right')
axes[1].axhline(y=50, color='red', linestyle='--', linewidth=1)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 7. Correlation Analysis

In [ ]:
# Create correlation matrix
merged_numeric = merged.copy()
merged_numeric['is_profitable'] = (merged_numeric['Closed PnL'] > 0).astype(int)
merged_numeric['is_buy'] = (merged_numeric['Side'] == 'BUY').astype(int)
merged_numeric['sentiment_score'] = merged_numeric['value']

corr_features = ['sentiment_score', 'Closed PnL', 'Size USD', 'is_profitable', 'is_buy', 'hour']
corr_matrix = merged_numeric[corr_features].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='RdYlGn', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix: Trading Metrics & Sentiment', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nKey Correlations:")
print(f"Sentiment Score vs PnL: {corr_matrix.loc['sentiment_score', 'Closed PnL']:.4f}")
print(f"Trade Size vs PnL: {corr_matrix.loc['Size USD', 'Closed PnL']:.4f}")
print(f"Hour vs PnL: {corr_matrix.loc['hour', 'Closed PnL']:.4f}")

## 8. Key Insights Summary

In [ ]:
print("="*80)
print("KEY INSIGHTS SUMMARY")
print("="*80)
print()
print("1. SENTIMENT IMPACT")
print("   - Extreme Greed shows highest average PnL and win rate (46.49%)")
print("   - Traders perform best during bullish sentiment periods")
print()
print("2. TEMPORAL PATTERNS")
print(f"   - Peak performance at hour {hourly_pnl.idxmax()} UTC (${hourly_pnl.max():.2f} avg PnL)")
print("   - Strong intraday variations suggest timing is crucial")
print()
print("3. TRADER CONCENTRATION")
print(f"   - {(trader_performance['Closed PnL'] > 0).mean()*100:.1f}% of traders are profitable")
print("   - Performance highly concentrated among top traders")
print()
print("4. ASSET FOCUS")
print(f"   - Top coin: {top_coins.index[0]} with {top_coins.iloc[0]:,} trades")
print("   - Platform token (HYPE) dominates trading activity")
print()
print("5. STRATEGIC OPPORTUNITIES")
print("   - Long bias during Extreme Greed periods")
print("   - Optimal trading windows identified")
print("   - Position sizing more important than frequency")
print("="*80)